# Assignment 3 Part A - Recurrent Neural Networks

Welcome to the first part of the third assignment!

We'll be covering the implementation of RNNs. The main ideas will be relatively understandable, but the implementations may get a little confusing. So read everything carefully!

In [ ]:
# Make sure to run this cell without modifying anything in it!
import numpy as np

# Import the code in `mytorch/nn.py`
from mytorch import nn

# Import function that checks correctness of your answers. 
from tests import compare_to_answer

# These iPython functions make it so imported code is automatically reimported here if they are changed
%load_ext autoreload
%autoreload 2

# Section 1 - RNN Cell


## Question 1.1 - `RNNCell.forward()`

We'll begin by implementing a basic Elman RNN cell.

In `mytorch/nn.py`, implement `RNNCell.forward()` using the following description:

<div style="text-align:center">
    <img src="images/rnn_cell.png" width="250"/>
</div>

Above is the a visualization of the cell. You can see the previous timestep's outputted hidden state $h_{t-1}$ on the top left, and the current timestep's input $x_t$ on the bottom. The green box is where our computations will happen.

Calculate and `return` $h_t$ like so:

$$\begin{align*}
    y_t &= x_t W^{T}_{ih}+b_{ih}+h_{(t-1)} W^{T}_{hh}+b_{hh} \\
    h_t &= \text{Tanh}(y_t)
\end{align*}$$

$$\begin{align*}
    &\text{Where $W^{T}_{ih}$ is a transposed weight matrix for processing the input data $x_t$ and} \\
    &\text{$W^{T}_{hh}$ is a transposed weight matrix for processing the previous hidden state $h_{(t-1)}$.} \\
    &\text{Tanh refers to the Tanh activation function.}
\end{align*}$$

**Notes**:
- We've provided you an implementation of `Tanh` in `nn.py`.
- `Tanh` is also already initialized for you in `RNNCell.__init__()` as `self.activation`.
    - When you want to use it, call `self.activation.forward()` with the appropriate input.
- You can refer to the official [torch docs](https://pytorch.org/docs/stable/generated/torch.nn.RNNCell.html) for this object if you want.

In [ ]:
def test_rnncell_forward_1(RNNCell):
    layer = RNNCell(3, 4)
    layer.weight_ih = np.array([[-3., -2., -1.],
                                [0., 1., 2.],
                                [3., 4., 5.],
                                [6., 7., 8.]])
    layer.bias_ih = np.array([[-1., -2., 3., 4.]])
    layer.weight_hh = np.array([[-8., -7., -6., -5.],
                                [-4., -3., -2., -1.],
                                [0., 1., 2., 3.],
                                [4., 5., 6., 7.]])
    layer.bias_hh = np.array([[1., 2., -3., -4.]])
    x = np.array([[1., 2., 3.],
                [4., 5., 6.]])
    h_prev = np.array([[1., 2., 3., 4.],
                    [3., 2., 1., 0.]])

    out = layer.forward(x, h_prev)
    return out

test_rnncell_forward_1(nn.RNNCell)

In [ ]:
from tests import test_rnncell_forward_1, test_rnncell_forward_2, test_rnncell_forward_3

answer_1 = test_rnncell_forward_1(nn.RNNCell)
answer_2 = test_rnncell_forward_2(nn.RNNCell)
answer_3 = test_rnncell_forward_3(nn.RNNCell)

If you passed the tests above, assign the string "Question 1 passed" to ans1 in the code cell below.

In [ ]:
### GRADED

ans1 = None

# YOUR CODE HERE
raise NotImplementedError()

## Question 1.2 - `RNNCell.backward()`

Now for backprop. There are actually *seven* gradients we need to calculate.

**Gradient of the loss w.r.t. input of activation**

First, we need to calculate the gradient w.r.t. the input of the `Tanh` activation. We need this to calculate the other gradients.

$$\nabla_{y_t} \text{Loss} = \nabla_{y_t} h_{t} \odot \nabla_{h_{t}} \text{Loss}$$

$$\begin{align*}
    &\text{Where $\nabla_{y_t} h_{t}$ is the gradient of the layer's output $h_t$ w.r.t. the input of the activation function $y_t$,} \\
    &\text{and $\odot$ is the element-wise multiplication operator.}
\end{align*}$$

Notes:
- You can use `self.activation.backward()`, but will have to give it a `state`.
    - The `state` is the output of `RNNCell.forward()` at the current timestep.

**Gradients of the loss w.r.t. the weights and biases at this timestep:**

Now we need to calculate and store these four gradients:

$$\begin{align*}
    \nabla_{W_{ih}} \text{Loss} += \nabla_{y_t} \text{Loss}^T h_{(t, l-1)} \\
    \nabla_{W_{hh}} \text{Loss} += \nabla_{y_t} \text{Loss}^T h_{(t-1, l)} \\
    \nabla_{b_{ih}} \text{Loss} += \sum_{n=0}^{N-1}{\nabla_{y_t} \text{Loss}_n} \\
    \nabla_{b_{hh}} \text{Loss} += \sum_{n=0}^{N-1}{\nabla_{y_t} \text{Loss}_n}
\end{align*}$$

Notes:
- The first two formulas use matrix multiplications.
- The last two formulas are simply summing across the `batch_size` axis, yielding a size `(hidden_size,)` array.
- The last two formulas have identical right hand sides on purpose.
    - Think about why this is; it has to do the partial derivatives w.r.t. each of those bias terms.
- Notice the `+=` in the above equations.
    - Similar to `nn.Conv1d` in assignment 2a, we use the weights and biases multiple times across different timesteps, so we need to accumulate their total influence over time.
    - Backprop will pass through this layer for each time step it was used.
        - It'll go in reverse order of time (last timestep -> first timestep)
        - By the first timestep, it'll have finished accumulating the weight/bias gradients that represent the params' total influence on the output 

**Gradients of the loss w.r.t. input and prev hidden state:**

Almost done! Calculate and `return` these two gradients:

$$\begin{align*}
\nabla_{x_t} \text{Loss} = \nabla_{y_t} \text{Loss} W_{ih}\\
\nabla_{h_{(t-1)}} \text{Loss} = \nabla_{y_t} \text{Loss} W_{hh}
\end{align*}$$

Notes:

- Notice that these use `=` instead of `+=`
    - These are not accumulated. We simply pass them backward for previous timesteps or for previous layers.

In [ ]:
def test_rnncell_backward_1(RNNCell):
    layer = RNNCell(3, 4)
    layer.weight_ih = np.array([[-0.01, -0.02, -0.03],
                                [0.01, 0.02, 0.03],
                                [0.04, 0.05, 0.06],
                                [0.07, 0.08, 0.09]])
    layer.bias_ih = np.array([[-0.01, -0.02, 0.03, 0.04]])
    layer.weight_hh = np.array([[-0.08, -0.07, -0.06, -0.05],
                                [-0.04, -0.03, -0.02, -0.01],
                                [0., 0.01, 0.02, 0.03],
                                [0.04, 0.05, 0.06, 0.07]])
    layer.bias_hh = np.array([[0.01, 0.02, -0.03, -0.04]])
    x = np.array([[1., 2., 3.],
                  [4., 5., 6.]])
    h_prev = np.array([[1., 2., 3., 4.],
                    [3., 2., 1., 0.]])
    
    grad = np.array([[ 0.402888  ,  0.01693988,  0.04669028,  0.54219921],
                     [-0.88253697,  0.11543817, -0.70723666, -0.69393529]])
    h_prev_l = np.array([[-0.62162275, -0.96164911,  0.21242244],
                         [ 0.27069328, -0.83331687, -0.6866582 ]])
    h_prev_t = np.array([[ 0.33983293,  0.93178091,  0.69990522, -0.15981829],
                         [ 0.12239934, -0.56387259, -0.9556718 , -0.31575391]])

    dx, dh = layer.backward(grad, h_prev, h_prev_l, h_prev_t)
    return dx, dh, layer.grad_weight_ih, layer.grad_weight_hh, layer.grad_bias_ih, layer.grad_bias_hh

test_rnncell_backward_1(nn.RNNCell)

In [ ]:
from tests import test_rnncell_backward_1, test_rnncell_backward_2, test_rnncell_backward_3

answer_1 = test_rnncell_backward_1(nn.RNNCell)
answer_2 = test_rnncell_backward_2(nn.RNNCell)
answer_3 = test_rnncell_backward_3(nn.RNNCell)

If you passed the tests above, assign the string "Question 2 passed" to ans2 in the code cell below.

In [ ]:
### GRADED

ans2 = None

# YOUR CODE HERE
raise NotImplementedError()

# Section 2 - `RNN` layer

The `RNN` object contains one or more `RNNCell`s and handles forward and backward prop over these cells.

- The number of cells is specified by the `num_layers` parameter.
- Documentation for the actual object is [here](https://pytorch.org/docs/stable/generated/torch.nn.RNN.html)
    - Assume `bidirectional=False` for this assignment.
        - In practice, bidirectionality is usually crucial for RNN-based layers. You'll even use a bidirectional RNN in the second part of this assignment.
        - We choose not to have you implement it because the idea of it is simple once you understand it, but the implementation is confusing/time-consuming.
        - For more understanding of bidirectionality, refer to lecture (or Google)!
    - Assume `batch_first=True` for this assignment.

## Question 2.1 - `RNN.forward()`


<div style="text-align:center">
    <img src="images/rnn_forward.png" width="350"/>
</div>

Above is a visualization of `RNN.forward()`. We're given an input with `seq_len=3`, and we have `num_layers=2` `RNNCell`s. The first layer (first `RNNCell`) is on the bottom, and is used **3 times**, once for each step of the input. 

Really, the main idea is to pass each timestep of the input through each cell of your RNN in order. That's it.

The hard part is indexing and carefully overwriting objects to make your code more efficient.

**Pseudocode**
- Declare a `hidden` array to store all hidden states in, and store the initial hidden state `h_0` in it (if given) 
- for each timestep `t`:
    - declare `x_t` by getting a slice of `x` at time `t` along the appropriate axis
        - `x_t` should be shaped `(batch_size, input_size)`
    - for each layer `l`:
        - set `x_t` to the output of the `l`th layer's forward pass
            - provide as inputs `x_t` and a slice of the `hiddens` vector at time `t` and layer index `l`
                - the `hiddens` vector slice should be shaped `(batch_size, hidden_size)`
        - store `x_t` in the `hiddens` vector at the `t+1`th time and the `l`th layer


In [ ]:
def test_rnn_forward_1(RNN):
    rnn = RNN(3, 4, num_layers=2)
    
    rnn.layers[0].weight_ih = np.array([[-0.01, -0.02, -0.03],
                                [0.01, 0.02, 0.03],
                                [0.04, 0.05, 0.06],
                                [0.07, 0.08, 0.09]])
    rnn.layers[0].bias_ih = np.array([[-0.01, -0.02, 0.03, 0.04]])
    
    rnn.layers[0].weight_hh = np.array([[-0.08, -0.07, -0.06, -0.05],
                                [-0.04, -0.03, -0.02, -0.01],
                                [0., 0.01, 0.02, 0.03],
                                [0.04, 0.05, 0.06, 0.07]])
    rnn.layers[0].bias_hh = np.array([[0.01, 0.02, -0.03, -0.04]])
    
    rnn.layers[1].weight_ih = np.array([[-0.48600084,  0.56108125,  0.60350991, -0.12398511],
       [-0.61426125,  0.74533839,  0.28829775,  0.39298129],
       [-0.23660597,  0.60180463, -0.17925026, -0.91010067],
       [ 0.32842213,  0.88810569,  0.47370219, -0.02676785]])
    rnn.layers[1].bias_ih = np.array([[ 0.80336632,  0.5080941 , -0.02041526, -0.23457551]])
    rnn.layers[1].weight_hh = np.array([[-0.66123341,  0.89060331,  0.52273452,  0.51176128],
       [ 0.56754451, -0.00703578,  0.7770004 ,  0.27559471],
       [ 0.55518779, -0.73064323,  0.9099351 ,  0.57656225],
       [ 0.36056783, -0.05294811,  0.47201524, -0.4093564 ]])
    rnn.layers[1].bias_hh = np.array([[-0.03656752, -0.7702112 ,  0.43095679,  0.89191036]])
    
    x = np.array([[[ 0.54963061, -0.05740221, -0.18798255],
        [-0.05139327,  0.31849613, -0.71975814],
        [ 0.25741578,  0.5026123 ,  0.64851701],
        [ 0.15833562,  0.45382923,  0.17070994],
        [-0.02513259,  0.7292614 ,  0.84537154]],

       [[ 0.96812713, -0.77282445, -0.95171846],
        [ 0.38731339, -0.83533687,  0.99131245],
        [ 0.04599233, -0.95944108,  0.23168402],
        [ 0.16541892,  0.13714912, -0.95989072],
        [ 0.51472085,  0.37945332,  0.4281448 ]]])
    h_0 = np.array([[[-0.71008325, -0.07277072,  0.78528585, -0.35171659],
        [ 0.21250086,  0.90839637, -0.7255993 ,  0.53047081]],

       [[-0.86511414, -0.94116341, -0.36088932,  0.69915346],
        [-0.53193134,  0.29990252, -0.27214273, -0.72523786]]])
    
    out, hiddens = rnn.forward(x, h_0)
    return out, hiddens

test_rnn_forward_1(nn.RNN)

In [ ]:
from tests import test_rnn_forward_1, test_rnn_forward_2, test_rnn_forward_3

answer_1 = test_rnn_forward_1(nn.RNN)
answer_2 = test_rnn_forward_2(nn.RNN)
answer_3 = test_rnn_forward_3(nn.RNN)

If you passed the tests above, assign the string "Question 3 passed" to ans3 in the code cell below.

In [ ]:
### GRADED

ans3 = None

# YOUR CODE HERE
raise NotImplementedError()

## Question 2.2 - RNN.backward()

This is the famous "[backpropagation through time](https://en.wikipedia.org/wiki/Backpropagation_through_time)" algorithm that was independently derived by hand in the 80's and 90's by multiple researchers. Although [automatic differentiation](https://en.wikipedia.org/wiki/Automatic_differentiation) software like torch's `autograd` and modern computers make calculating arbitrary derivatives trivial at this point in time. 

The challenge comes from all of the information being passed around and stored. Multiple timesteps, multiple layers, and hidden states for each of these layers and timesteps. You essentially need to retrace the forward diagram in reverse order, passing every necessary tensor at the right step.

The indexing and explanation for this is truly unbearable. We've tried to simplify this, but the algorithm is just inherently messy.

**Pseudocode**
- Initialize `dx` and `dh_0`, the gradients of the input and initial hidden state.
    - Assign the last layer index of `dh_0` to be `grad` (given)
- for each timestep `t` in reverse order from `seq_len` to `1` (inclusive on both sides, see notes):
    - for each layer `l` in reverse order from `num_layers-1` to `1` (inclusive on both sides):
        - get `dx_t_l` and `dh_0[l]` from the backward pass of the `l`th layer
            - provide `dh_0[l]`, `hiddens[t][l]`, `hiddens[t][l-1]`, and `hiddens[t-1][l]` as inputs
                - See the docstring of `RNNCell.backward()` for what these are
            - we're calculating the gradient of loss w.r.t. the current cell
        - add `dx_t_l` to `dh_0[l-1]` using a `+=` operation
            - because the hidden state from the previous layer impacted the current cell, we need to add its gradient (its impact) to the gradient of the loss w.r.t. previous layer's hidden state
    - get `dx_t` and `dh_0[0]` from the backward pass of the `0`th layer
        - provide `dh_0[0]`, `hiddens[t][0]`, `x[:,t-1,:]`, and `hiddens[t-1][0]` as inputs
    - set `dx[:,t-1,:]` equal to `dx_t`
        - after making it back through all the cells, we're finally back at the input

**Notes:**
- By "inclusive on both sides", we mean `t`$\in$`[seq_len, seq_len-1, ..., 1]`.
    - For `l` it'd be `l`$\in$`[num_layers-1, num_layers-2, ..., 1]`.
- The reason we end the inner loop at `1` is because the `0`th layer has to be given a slice from `x`.
    - Notice that we're giving this slice of `x` as `h_prev_l` in `RNNCell.backward()`. This is the "previous layer", which doesn't exist before the `0`th layer. So we just give it the original input at that timestep.
    - If you can find a way to neatly merge this into the `for` loop, hats off to you.
- Notice: `RNNCell.backward()` handles storing gradients for us

In [ ]:
def test_rnn_backward_1(RNN):
    rnn = RNN(3, 4, num_layers=2)
    
    rnn.layers[0].weight_ih = np.array([[-0.01, -0.02, -0.03],
                                [0.01, 0.02, 0.03],
                                [0.04, 0.05, 0.06],
                                [0.07, 0.08, 0.09]])
    rnn.layers[0].bias_ih = np.array([[-0.01, -0.02, 0.03, 0.04]])
    
    rnn.layers[0].weight_hh = np.array([[-0.08, -0.07, -0.06, -0.05],
                                [-0.04, -0.03, -0.02, -0.01],
                                [0., 0.01, 0.02, 0.03],
                                [0.04, 0.05, 0.06, 0.07]])
    rnn.layers[0].bias_hh = np.array([[0.01, 0.02, -0.03, -0.04]])
    
    rnn.layers[1].weight_ih = np.array([[-0.48600084,  0.56108125,  0.60350991, -0.12398511],
       [-0.61426125,  0.74533839,  0.28829775,  0.39298129],
       [-0.23660597,  0.60180463, -0.17925026, -0.91010067],
       [ 0.32842213,  0.88810569,  0.47370219, -0.02676785]])
    rnn.layers[1].bias_ih = np.array([[ 0.80336632,  0.5080941 , -0.02041526, -0.23457551]])
    rnn.layers[1].weight_hh = np.array([[-0.66123341,  0.89060331,  0.52273452,  0.51176128],
       [ 0.56754451, -0.00703578,  0.7770004 ,  0.27559471],
       [ 0.55518779, -0.73064323,  0.9099351 ,  0.57656225],
       [ 0.36056783, -0.05294811,  0.47201524, -0.4093564 ]])
    rnn.layers[1].bias_hh = np.array([[-0.03656752, -0.7702112 ,  0.43095679,  0.89191036]])
    
    x = np.array([[[ 0.54963061, -0.05740221, -0.18798255],
        [-0.05139327,  0.31849613, -0.71975814],
        [ 0.25741578,  0.5026123 ,  0.64851701],
        [ 0.15833562,  0.45382923,  0.17070994],
        [-0.02513259,  0.7292614 ,  0.84537154]],

       [[ 0.96812713, -0.77282445, -0.95171846],
        [ 0.38731339, -0.83533687,  0.99131245],
        [ 0.04599233, -0.95944108,  0.23168402],
        [ 0.16541892,  0.13714912, -0.95989072],
        [ 0.51472085,  0.37945332,  0.4281448 ]]])
    h_0 = np.array([[[-0.71008325, -0.07277072,  0.78528585, -0.35171659],
        [ 0.21250086,  0.90839637, -0.7255993 ,  0.53047081]],

       [[-0.86511414, -0.94116341, -0.36088932,  0.69915346],
        [-0.53193134,  0.29990252, -0.27214273, -0.72523786]]])
    
    rnn.forward(x, h_0)
    
    grad = np.array([[-0.1, 0.2, -0.3, 0.4],
                     [-0.4, 0.5, -0.6, 0.7]])
    
    dx, dh_0 = rnn.backward(grad)
    
    # Gather all the gradients in a single list
    layer_gradients = []
    for n in range(rnn.num_layers):
        layer_gradients.append([rnn.layers[n].grad_weight_ih, rnn.layers[n].grad_weight_hh, rnn.layers[n].grad_bias_ih, rnn.layers[n].grad_bias_hh])
    
    return dx, dh_0, layer_gradients

# We check the input and initial hidden state gradients, as well as gradients on every RNNCell
test_rnn_backward_1(nn.RNN)

In [ ]:
from tests import test_rnn_backward_1, test_rnn_backward_2, test_rnn_backward_3

answer_1 = test_rnn_backward_1(nn.RNN)
answer_2 = test_rnn_backward_2(nn.RNN)
answer_3 = test_rnn_backward_3(nn.RNN)

If you passed the tests above, assign the string "Question 4 passed" to ans4 in the code cell below.

In [ ]:
### GRADED

ans4 = None

# YOUR CODE HERE
raise NotImplementedError()

# Epilogue - Part A's

Good work on completing all three 'Part A' portions of each assignemnt!

The goal of each 'Part A' was really to convince you that deep learning is just math at work. The perception that deep learning is a completely opaque black box is misguided at best and harmful at worst. We hope you'll agree that the math and code wasn't fancy - just matrix operations, derivatives, and simple loops.

If you consolidate all the code for `mytorch` you wrote throughout the course, you have a basic functioning deep learning library. If the course had more time, you'd have been able to add modules like `Dropout`, the `Adam` optimizer, different convolutions, `LSTM`, etc. We encourage you to try implementing those yourself in this same library. You can look up descriptions of how they work, and try to recreate them. To check if your implementation is correct, compare it to the actual torch's implementation. That's essentially how we developed this library.

The actual PyTorch differs in one big way though - while we manually implemented backpropagation and `backward()` methods for this course, PyTorch actually often automates this process using [autograd](https://pytorch.org/tutorials/beginner/blitz/autograd_tutorial.html), a package for [automatic differentiation](https://en.wikipedia.org/wiki/Automatic_differentiation).

For details on how `autograd` works, read the links provided. For the second link, note that `autograd` uses "reverse accumulation". The reason we use this instead of forward is explained in the "reverse accumulation" section of that link.

But we're not quite done yet - the second part of this assignment still remains. We'll see you there.